# Part2: RAG (Retrieval-Augmented Generation)

## 0. 환경 구성

### 1) 라이브러리 설치

In [1]:
!pip install -q langchain langchain-groq langchain-community langchain-chroma sentence-transformers chromadb langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [2]:
import langchain

langchain.__version__

'1.3.11'

### 2) Groq 인증키 설정

In [3]:
from google.colab import userdata
import os

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

## 1. RAG 파이프라인 개요

- Load Data - Text Split - Indexing - Retrieval - Generation

STEP 1 : Load Data

- loader = WebBaseLoader(url) → "이 주소로 가서 읽어올 준비를 하는 것" (아직 안 감)
- docs = loader.load() → "실제로 가서 읽고, 읽은 내용을 문서 형태로 정리해서 가져오는 것"

In [4]:
# Data Loader - 웹페이지 데이터 가져오기
from langchain_community.document_loaders import WebBaseLoader

# 위키피디아 정책과 지침
url = 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8'
loader = WebBaseLoader(url)

# 웹페이지 텍스트 -> Documents
docs = loader.load()

print(len(docs))
print(len(docs[0].page_content))
print(docs[0].page_content[5000:6000])

/tmp/ipykernel_532/2216952504.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


1
13329
요.
특정 사용자가 공동체의 규범을 총체적으로 어기고 있다면 규범 준수를 위해 좀 더 빠르게 강력한 수단을 이용해야 합니다. 특히 정책 문서에 명시된 원칙을 지키지 않는 것은 대부분의 경우 다른 사용자에게 받아들여지지 않습니다 (다른 분들에게 예외 상황임을 설득할 수 있다면 가능하기는 하지만요). 이는 당신을 포함해서 편집자 개개인이 정책과 지침을 직접 집행 및 적용한다는 것을 의미합니다.
특정 사용자가 명백히 정책에 반하는 행동을 하거나 정책과 상충되는 방식으로 지침을 어기는 경우, 특히 의도적이고 지속적으로 그런 행위를 하는 경우 해당 사용자는 관리자의 제재 조치로 일시적, 혹은 영구적으로 편집이 차단될 수 있습니다. 영어판을 비롯한 타 언어판에서는 일반적인 분쟁 해결 절차로 끝낼 수 없는 사안은 중재위원회가 개입하기도 합니다.
문서 내용
정책과 지침의 문서 내용은 처음 읽는 사용자라도 원칙과 규범을 잘 이해할 수 있도록 다음 원칙을 지켜야 합니다.
명확하게 작성하세요. 소수만 알아듣거나 준법률적인 단어, 혹은 지나치게 단순한 표현은 피해야 합니다. 명확하고, 직접적이고, 모호하지 않고, 구체적으로 작성하세요. 지나치게 상투적인 표현이나 일반론은 피하세요. 지침, 도움말 문서 및 기타 정보문 문서에서도 "해야 합니다" 혹은 "하지 말아야 합니다" 같이 직접적인 표현을 굳이 꺼릴 필요는 없습니다.
가능한 간결하게, 너무 단순하지는 않게. 정책이 중언부언하면 오해를 부릅니다. 불필요한 말은 생략하세요. 직접적이고 간결한 설명이 마구잡이식 예시 나열보다 더 이해하기 쉽습니다. 각주나 관련 문서 링크를 이용하여 더 상세히 설명할 수도 있습니다.
규칙을 만든 의도를 강조하세요. 사용자들이 상식대로 행동하리라 기대하세요. 정책의 의도가 명료하다면, 추가 설명은 필요 없죠. 즉 규칙을 '어떻게' 지키는지와 더불어 '왜' 지켜야 하는지 확실하게 밝혀야 합니다.
범위는 분명히, 중복은 피하기. 되도록 앞부분에서 정책 및 지침의 목적과 범위를 분명하게 밝혀야 

STEP 2: Split tests

RecursiveCharacterTextSplitter 클래스는 텍스트를 재귀적으로 분할하는데 이때 분할된 청크들이 설정된 chunk_size보다 작아질 때까지 이 과정을 반복하여 의미적으로 관련 있는 텍스트 조각들이 같이 있도록 한다.

In [5]:
# Text Split (Documents -> small chunks: Documents)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1000자씩 자르되 다음 청크가 이전 청크의 마지막 200자 포함 -> 오버랩이 없으면 문장이 갑자기 끊겨서 LLM에 넘기므로 답변이 이상해짐
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(len(splits))
print(splits[10])

18
page_content=''제안'은 완전 새로운 원칙이라기보다, 기존의 불문율이나 토론 총의의 문서를 통한 구체화에 가깝습니다. 많은 사람들이 쉽게 제안을 받아들이도록 하기 위해서는, 기초적인 원칙을 우선 정하고 기본 틀을 짜야 합니다. 정책과 지침의 기본 원칙은 "왜 지켜야 하는가?", "어떻게 지켜야 하는가?" 두 가지입니다. 특정 원칙을 정책이나 지침으로 확립하기 위해서는 우선 저 두 가지 물음에 성실하게 답하는 제안 문서를 작성해야 합니다.
좋은 아이디어를 싣기 위해 사랑방이나 관련 위키프로젝트에 도움을 구해 피드백을 요청할 수 있습니다. 이 과정에서 공동체가 어느 정도 받아들일 수 있는 원칙이 구체화됩니다. 많은 이와의 토론을 통해 공감대가 형성되고 제안을 개선할 수 있습니다.
정책이나 지침은 위키백과 내의 모든 편집자들에게 적용되는 원칙이므로 높은 수준의 총의가 요구됩니다. 제안 문서가 잘 짜여졌고 충분히 논의되었다면, 더 많은 공동체의 편집자와 논의를 하기 위해 승격 제안을 올려야 합니다. 제안 문서 맨 위에 {{제안}}을 붙여 제안 안건임을 알려주고, 토론 문서에 {{의견 요청}}을 붙인 뒤 채택 제안에 관한 토론 문단을 새로 만들면 됩니다. 많은 편집자들에게 알리기 위해 관련 내용을 {{위키백과 소식}}에 올리고 사랑방에 이를 공지해야 하며, 합의가 있을 경우 미디어위키의 sitenotice(위키백과 최상단에 노출되는 구역)에 공지할 수도 있습니다. 
차후 공지가 불충분했다는 이의 제기를 피하려면, 위의 링크를 이용하여 공지하세요. 공지에 비중립적인 단어를 사용하는 등의 선전 행위는 피하세요.' metadata={'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'k

In [6]:
splits[10].page_content

'\'제안\'은 완전 새로운 원칙이라기보다, 기존의 불문율이나 토론 총의의 문서를 통한 구체화에 가깝습니다. 많은 사람들이 쉽게 제안을 받아들이도록 하기 위해서는, 기초적인 원칙을 우선 정하고 기본 틀을 짜야 합니다. 정책과 지침의 기본 원칙은 "왜 지켜야 하는가?", "어떻게 지켜야 하는가?" 두 가지입니다. 특정 원칙을 정책이나 지침으로 확립하기 위해서는 우선 저 두 가지 물음에 성실하게 답하는 제안 문서를 작성해야 합니다.\n좋은 아이디어를 싣기 위해 사랑방이나 관련 위키프로젝트에 도움을 구해 피드백을 요청할 수 있습니다. 이 과정에서 공동체가 어느 정도 받아들일 수 있는 원칙이 구체화됩니다. 많은 이와의 토론을 통해 공감대가 형성되고 제안을 개선할 수 있습니다.\n정책이나 지침은 위키백과 내의 모든 편집자들에게 적용되는 원칙이므로 높은 수준의 총의가 요구됩니다. 제안 문서가 잘 짜여졌고 충분히 논의되었다면, 더 많은 공동체의 편집자와 논의를 하기 위해 승격 제안을 올려야 합니다. 제안 문서 맨 위에 {{제안}}을 붙여 제안 안건임을 알려주고, 토론 문서에 {{의견 요청}}을 붙인 뒤 채택 제안에 관한 토론 문단을 새로 만들면 됩니다. 많은 편집자들에게 알리기 위해 관련 내용을 {{위키백과 소식}}에 올리고 사랑방에 이를 공지해야 하며, 합의가 있을 경우 미디어위키의 sitenotice(위키백과 최상단에 노출되는 구역)에 공지할 수도 있습니다. \n차후 공지가 불충분했다는 이의 제기를 피하려면, 위의 링크를 이용하여 공지하세요. 공지에 비중립적인 단어를 사용하는 등의 선전 행위는 피하세요.'

In [7]:
splits[10].metadata

{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8',
 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전',
 'language': 'ko'}

Step 3: Indexing

Indexing (Texts -> Embedding -> Store)

In [8]:
# 임베딩 모델 로드 (jhgan/ko-sroberta-multitask는 한국어 문장 임베딩에 특화된 모델)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask"
)

/tmp/ipykernel_532/1307353551.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

- Chroma라는 벡터 데이터베이스(Vector DB) 사용
- similarity_search()는 기본적으로 가장 관련성 높은 상위 4개(k=4)만 반환

In [9]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings,
                                    collection_name="ko_sroberta_collection")
docs = vectorstore.similarity_search("격하 과정에 대해서 설명해주세요.")
print(len(docs))
print(docs[0].page_content)

4
차후 공지가 불충분했다는 이의 제기를 피하려면, 위의 링크를 이용하여 공지하세요. 공지에 비중립적인 단어를 사용하는 등의 선전 행위는 피하세요.
토론이 끝났다면 선언과 함께 {{토론보존}} 틀을 이용하여 닫습니다. 총의 판단은 여타 토론과 마찬가지로  분쟁 해결 정책에서 갈음해 처리합니다. 토론을 통해 정책이나 지침 채택 여부를 논의하며, 이 과정에서 제안 문서가 크게 수정될 수도 있습니다. 토론 중 제안을 정식 정책/지침으로 채택하자는 합의로 모이고 나서 2주 (정확히 14일. 이후 내용은 모두 같습니다) 간 제안을 대폭 수정해야 하는 변경안 제시나 명확한 근거가 존재하는 반대가 나오지 않는다면 정책이나 지침으로 정식으로 채택됩니다. 반대로 토론자들 사이에서 채택을 거부한다는 합의가 모아져서 2주간 명확한 근거가 존재하는 반대 의견이 나오지 않는다면 채택안 거부 총의가 모아졌다고 보아 기각됩니다. 주요한 총의 판단 기준은 다음과 같습니다. 
정책과 지침 관련 총의는 추세가 뚜렷해야 하지만, 만장일치일 필요는 없습니다.
제안 작성자 이외에도 공동체 전체에 노출되어 있어야 합니다.
제안 문서의 영향력을 고려하세요.
토론 중 중대한 문제가 제기되었는가?
다른 정책 또는 지침과 모순되지는 않는가?
새로 제정된 다른 정책 또는 지침과 병합될 수도 있는가?
이미 존재하는 정책 및 지침과 중복되지는 않는가?
제안 처리는 단순 득표수로 결정되지 않습니다. 투표는 토론을 대체하지 않으며, 득표수가 총의와 합치하는 것도 아닙니다.
총의가 불분명하고, 이 상황이 개선될 것 같지 않다면 이는 거부됩니다.
채택 토론은 세 가지로 귀결됩니다. 채택, 총의 없음, 거부.  토론 종결 이후 {{제안}} 틀을 제거하고 알맞은 틀을 삽입하세요. 가령 {{정책}}, {{지침}}, {{수필}}, {{거부}}  등을 말이죠. 그 외에 다양한 틀에는 위키백과 이름공간 관련 틀 (영어 위키백과)을 참고하세요.


위 결과는 본문 자체에 "격하 과정은 채택 과정과 비슷합니다"라는 문장이 있으므로 합리적인 답변으로 생각됌

In [11]:
for i, doc in enumerate(docs):
    print(f"--- 결과 {i} ---")
    print(doc.page_content[:200])
    print()

--- 결과 0 ---
차후 공지가 불충분했다는 이의 제기를 피하려면, 위의 링크를 이용하여 공지하세요. 공지에 비중립적인 단어를 사용하는 등의 선전 행위는 피하세요.
토론이 끝났다면 선언과 함께 {{토론보존}} 틀을 이용하여 닫습니다. 총의 판단은 여타 토론과 마찬가지로  분쟁 해결 정책에서 갈음해 처리합니다. 토론을 통해 정책이나 지침 채택 여부를 논의하며, 이 과정에서 제안

--- 결과 1 ---
정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히지 않았다면 더더욱이요.
격하
특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸

--- 결과 2 ---
아래에서 보듯, 당신은 과감하게 삽입하거나, 토론을 통하여 광범위한 총의를 모으는 방식으로 모범적인 활동 규범을 제시할 수 있습니다.
실질적인 변경
실행하세요. 정책 및 지침 문서를 실질적으로 변경하기 전에, 기존 관행에 대한 합리적인 예외를 설정하는 것이 유용할 수 있습니다. 이러한 방식으로 기존의 관행을 갱신하기 위해, 백:과감과 백:무시 정신에 따라 

--- 결과 3 ---
채택 토론은 세 가지로 귀결됩니다. 채택, 총의 없음, 거부.  토론 종결 이후 {{제안}} 틀을 제거하고 알맞은 틀을 삽입하세요. 가령 {{정책}}, {{지침}}, {{수필}}, {{거부}}  등을 말이죠. 그 외에 다양한 틀에는 위키백과 이름공간 관련 틀 (영어 위키백과)을 참고하세요.
제안이 통과되었다면, 위키백과:사랑방에 통과된 정책의 주요 내용을 



In [12]:
print(docs[1].page_content)

정책 및 지침을 과감히 편집할 시 되돌리기를 당했다면 먼저 되돌리기 하는 것 보다, 토론을 열어 의견을 나누는 것이 좋습니다. 토론 중 자신의 주장을 뒷받침하기 위한 정책 편집은 체제를 시험하는 것으로 보일 수 있습니다. 특히 편집 시 논쟁에 자신이 참여한 사실을 밝히지 않았다면 더더욱이요.
격하
특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸모없어질 수 있고, 다른 문서가 개선되어 내용이 중복될 수 있으며, 불필요한 내용이 증식할 수도 있습니다. 이 경우 편집자들은 정책을 지침으로 격하하거나, 정책 또는 지침을 보충 설명, 정보문, 수필 또는 중단 문서로 격하할 것을 제안할 수 있습니다. 
격하 과정은 채택 과정과 비슷합니다. 일반적으로 토론 문서 내 논의가 시작되고 프로젝트 문서 상단에 {{새로운 토론|문단=진행 중인 토론 문단}} 틀을 붙여 공동체의 참여를 요청합니다. 논의가 충분히 이루어진 후, 제3의 편집자가 토론을 종료하고 평가한 후 상태 변경 총의가 형성되었는지 판단해야 합니다. 폐지된 정책이나 지침은 최상단에 {{중단}} 틀을 붙여 더 이상 사용하지 않는 정책/지침임을 알립니다.
소수의 공동체 인원만 지지하는 수필, 정보문 및 기타 비공식 문서는 일반적으로 주된 작성자의 사용자 이름공간으로 이동합니다. 이러한 논의는 일반적으로 해당 문서의 토론란에서 이루어지며, 간혹 위키백과:의견 요청을 통해 처리되기도 합니다.
같이 보기
위키백과:위키백과의 정책과 지침 목록
위키백과:의견 요청
수필
위키백과:제품, 절차, 정책
위키백과:위키백과 공동체의 기대와 규범
기타 링크
위키백과:사랑방 (정책) - 현재는 폐지됨.
외부 링크
위키미디어 재단의 사명
위키미디어 재단이 추구하는 가치
meta:Founding principles - 메타위키의 창립원리
vte위키백과의 주요 정책과 지침 (?)
다섯 원칙
규칙에 얽매이지 마세요
컨텐츠 (?)P
확인 가능
독자 연구 금지
중립적 시각
위키백과에 대한 오해
생존 인물의 전기
저작권
G
문서 등재 기준


Step 4: Retrieval ~ Generation

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# LLM
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Rretriever
retriever = vectorstore.as_retriever()

# Combine Documents
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG Chain 연결
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# Chain 실행
rag_chain.invoke("격하 과정에 대해서 설명해주세요.")

'격하 과정은 특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸모없어질 수 있고, 다른 문서가 개선되어 내용이 중복될 수 있으며, 불필요한 내용이 증식할 수도 있는 경우에 해당 정책이나 지침을 지침으로 격하하거나, 정책 또는 지침을 보충 설명, 정보문, 수필 또는 중단 문서로 격하하는 것을 말합니다.\n\n격하 과정은 채택 과정과 비슷합니다. 일반적으로 토론 문서 내 논의가 시작되고 프로젝트 문서 상단에 {{새로운 토론|문단=진행 중인 토론 문단}} 틀을 붙여 공동체의 참여를 요청합니다. 논의가 충분히 이루어진 후, 제3의 편집자가 토론을 종료하고 평가한 후 상태 변경 총의가 형성되었는지 판단해야 합니다. 폐지된 정책이나 지침은 최상단에 {{중단}} 틀을 붙여 더 이상 사용하지 않는 정책/지침임을 알립니다.\n\n소수의 공동체 인원만 지지하는 수필, 정보문 및 기타 비공식 문서는 일반적으로 주된 작성자의 사용자 이름공간으로 이동합니다. 이러한 논의는 일반적으로 해당 문서의 토론란에서 이루어지며, 간혹 위키백과:의견 요청을 통해 처리되기도 합니다.'

## 2. Agent 실행

위에서는 "질문 → 무조건 검색 → 답변" 고정된 흐름(체인)

create_agent를 쓰면, LLM이 "이 질문에 검색이 필요한가?"를 스스로 판단하고, 필요하면 도구를 호출, 최종 답변을 생성할 때까지 반복적으로 추론과 행동을 수행

이러한 패턴을 ReAct(Reasoning + Acting)

In [19]:
from langchain.agents import create_agent
from langchain_core.tools.retriever import create_retriever_tool
from langchain_community.vectorstores import Chroma

from langchain_groq import ChatGroq

# 1. 기존 Chroma vectorstore를 retriever로 변환
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. retriever를 도구로 감싸기
retriever_tool = create_retriever_tool(
    retriever,
    name="wikidocs_search",
    description="위키독스 문서에서 관련 내용을 검색할 때 사용하세요. "
                "질문이 문서 내용과 관련 있으면 반드시 먼저 이 도구를 호출하세요."
)

# 3. LLM 설정 (기존에 쓰시던 Groq 그대로)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# 4. 에이전트 생성
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
  system_prompt = (
    "당신은 위키독스 문서를 기반으로 질문에 답하는 어시스턴트입니다. "
    "검색 도구로 찾은 문서 내용을 반드시 구체적으로 인용·설명하며 답변하세요. "
    "일반론이나 메타 설명(예: '~에 대한 내용이 있습니다') 대신, "
    "실제 절차·정의·항목을 상세히 풀어서 답하세요. "
    "문서에 없는 내용은 모른다고 답하세요.")
)

# 5. 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "격하 과정에 대해서 설명해주세요."}]
})

print(result["messages"][-1].content)

위키백과:정책과 지침 문서에서, 격하 과정은 특정 정책이나 지침이 편집 관행이나 공동체 규범이 바뀌며 쓸모없어질 수 있고, 다른 문서가 개선되어 내용이 중복될 수 있으며, 불필요한 내용이 증식할 수도 있습니다. 이 경우 편집자들은 정책을 지침으로 격하하거나, 정책 또는 지침을 보충 설명, 정보문, 수필 또는 중단 문서로 격하할 것을 제안할 수 있습니다. 격하 과정은 채택 과정과 비슷합니다. 일반적으로 토론 문서 내 논의가 시작되고 프로젝트 문서 상단에 {{새로운 토론|문단=진행 중인 토론 문단}} 틀을 붙여 공동체의 참여를 요청합니다. 논의가 충분히 이루어진 후, 제3의 편집자가 토론을 종료하고 평가한 후 상태 변경 총의가 형성되었는지 판단해야 합니다. 폐지된 정책이나 지침은 최상단에 {{중단}} 틀을 붙여 더 이상 사용하지 않는 정책/지침임을 알립니다. 소수의 공동체 인원만 지지하는 수필, 정보문 및 기타 비공식 문서는 일반적으로 주된 작성자의 사용자 이름공간으로 이동합니다. 이러한 논의는 일반적으로 해당 문서의 토론란에서 이루어지며, 간혹 위키백과:의견 요청을 통해 처리되기도 합니다.
